# 🤖 Notebook 03 — Model Development & Evaluation
## Comparaison de 6 Algorithmes de Clustering — De l'Espace Feature aux Segments Clients

---

## Section 0 — Contexte, Objectifs & Contraintes

Le notebook preprocessing (`02_preprocessing_fe.ipynb`) a produit un **feature store normalisé** de 9 features par client, prêt pour le clustering. Ce notebook compare **6 algorithmes** couvrant les 4 familles de clustering, sélectionne le meilleur modèle, valide les hypothèses EDA, et exporte les artefacts pour le dashboard.

### Les 6 Algorithmes et leur Justification

| Famille | Algorithme | Hyperparamètre clé | Justification |
|---------|-----------|-------------------|---------------|
| **Centroïdes** | K-Means | k (Elbow + Silhouette) | Baseline standard, rapide, scalable |
| **Centroïdes** | Bisecting K-Means | k (même critères) | Clusters plus équilibrés en taille (divisif) |
| **Hiérarchique** | CAH (Ward) | k (dendrogramme) | Pas de k a priori, validation visuelle |
| **Probabiliste** | GMM | k (BIC/AIC) | Clusters ellipsoïdaux, soft membership |
| **Densité** | DBSCAN | eps, min_samples | Clusters non-convexes, détecte le bruit |
| **Densité** | HDBSCAN | min_cluster_size | Densités variables, plus robuste que DBSCAN |

### Pipeline de ce Notebook

```
customer_features_scaled.parquet  (9 features, n_customers)
         │
         ▼
[Sections 1–6]  Hyperparameter Search × 6 algorithmes  → MLFlow child runs
         │
         ▼
[Section 7]     Final fits × 6 algorithmes              → MLFlow final runs
         │
         ▼
[Section 8]     Tableau de comparaison + composite score → Sélection BEST_ALGORITHM
         │
         ▼
[Section 9]     Profiling des clusters (radar, heatmap, PCA, nommage marketing)
         │
         ▼
[Section 10]    Validation H1–H6 (hypothèses EDA)
         │
         ▼
[Section 11]    Export : model.pkl + labeled.parquet + profile.parquet + MLFlow artefacts
```

> **⚠️ Règle de sélection finale :** Seuls les algorithmes **avec `predict()`** sont éligibles pour le modèle de production (KMeans, BisectingKMeans, CAH, GMM). DBSCAN et HDBSCAN sont **transductifs** — inclus dans la comparaison mais exclus de la sélection.

> **📦 Décision de production :** Suite à l'exploration de ce notebook, le pipeline de production () utilise une approche simplifiée **pure RFM (3 features : Recency, Frequency, Monetary)** avec IQR filtering (q5/q95) et normalisation log10. Cette réduction dimensionnelle porte le Silhouette Score de ~0.21 (9 features) à **0.449** grâce au pipeline UMAP(n_components=2, n_neighbors=750, min_dist=0) → KMeans k=4. Les 4 segments retenus : Acheteurs Budget, Dormants Budget, Acheteurs Premium, Dormants Premium — segmentation 2×2 Récence × Montant.

In [1]:
import sys
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # Force non-interactive backend — required for nbconvert (no display)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
import sklearn
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# ── Chemins projet — robuste nbconvert (cwd=project root) ET Jupyter (cwd=notebooks/) ──
_cwd = Path(os.getcwd())
PROJECT_ROOT  = _cwd.parent if _cwd.name == 'notebooks' else _cwd
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR    = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from features import FINAL_FEATURES
from model_utils import (
    compute_clustering_metrics,
    run_kmeans_search, fit_kmeans,
    run_bisecting_kmeans_search, fit_bisecting_kmeans,
    run_cah_search, fit_cah,
    run_gmm_search, fit_gmm,
    estimate_dbscan_eps, run_dbscan_search, fit_dbscan,
    run_hdbscan_search, fit_hdbscan,
    plot_elbow, plot_bic_aic, plot_silhouette_comparison,
    plot_metrics_comparison, plot_dendrogram,
    plot_cluster_heatmap, plot_radar_chart, plot_pca_clusters,
    build_cluster_profile, validate_hypotheses,
    save_model, load_model,
)

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'PROCESSED_DIR : {PROCESSED_DIR}')
print(f'sklearn version : {sklearn.__version__}')
print(f'MLFlow version  : {mlflow.__version__}')

PROJECT_ROOT  : /Users/mac/Downloads/School Projects/olist-customer-segmentation
PROCESSED_DIR : /Users/mac/Downloads/School Projects/olist-customer-segmentation/data/processed
sklearn version : 1.6.1
MLFlow version  : 3.1.4


In [2]:
# ── Chargement des feature stores ─────────────────────────────────────────────
X_scaled_df = pd.read_parquet(PROCESSED_DIR / 'customer_features_scaled.parquet')
if 'customer_unique_id' in X_scaled_df.columns:
    X_scaled_df = X_scaled_df.set_index('customer_unique_id')

df_raw = pd.read_parquet(PROCESSED_DIR / 'customer_features_raw.parquet')

# Alignement : les deux fichiers doivent être dans le même ordre
assert list(df_raw['customer_unique_id']) == list(X_scaled_df.index), \
    'Désalignement entre df_raw et X_scaled_df — vérifier notebook 02'

# Matrice numpy pour sklearn
X = X_scaled_df[FINAL_FEATURES].values

assert not np.isnan(X).any(), 'NaN détectés dans X — vérifier notebook 02'
assert X.shape[1] == 9, f'Attendu 9 features, obtenu {X.shape[1]}'

print(f'✅ Feature matrix chargée : {X.shape[0]:,} clients × {X.shape[1]} features')
print(f'   Parquet raw    : {df_raw.shape}')
print(f'   Parquet scaled : {X_scaled_df.shape}')
print(f'\nFINAL_FEATURES = {FINAL_FEATURES}')

✅ Feature matrix chargée : 93,358 clients × 9 features
   Parquet raw    : (93358, 21)
   Parquet scaled : (93358, 9)

FINAL_FEATURES = ['Log_Recency', 'Log_Monetary', 'Frequency_flag', 'avg_freight_ratio', 'avg_delivery_delay', 'avg_review_score', 'payment_type_cc_flag', 'avg_installments', 'region_freight_score']


In [3]:
# ── Setup MLFlow ──────────────────────────────────────────────────────────────
EXPERIMENT_NAME = 'olist_customer_segmentation'
mlflow.set_tracking_uri(str(PROJECT_ROOT / 'mlruns'))  # str() obligatoire (pas Path)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'✅ MLFlow configuré')
print(f'   Experiment   : {EXPERIMENT_NAME}')
print(f'   Tracking URI : {mlflow.get_tracking_uri()}')

✅ MLFlow configuré
   Experiment   : olist_customer_segmentation
   Tracking URI : /Users/mac/Downloads/School Projects/olist-customer-segmentation/mlruns


## Section 1 — K-Means : Elbow + Silhouette

**Pourquoi K-Means ?** Baseline incontournable pour la segmentation client. KMeans++ (init par défaut sklearn) garantit une meilleure initialisation que le random init, crucial sur un espace 9D avec un segment loyal minoritaire (~5%). On utilise `n_init=20` (2× le défaut) pour la stabilité.

**Lecture de l'Elbow :** Le WCSS décroît toujours avec k — l'inflexion (coude) indique le k optimal. Après l'inflexion, chaque cluster supplémentaire apporte peu de réduction de variance.

**Lecture du Silhouette :** Score entre -1 et +1. Un pic indique que les clusters sont denses et bien séparés. Sur données RFM, des valeurs de 0.15–0.30 sont typiques.

In [4]:
# Garde-fou : fermer tout run MLFlow dangling avant de démarrer
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name='kmeans_k_search') as parent_run:
    mlflow.set_tag('algorithm', 'kmeans')
    mlflow.set_tag('run_type', 'k_search')
    results_km = run_kmeans_search(
        X, k_range=range(2, 13), n_init=20,
        parent_run_id=parent_run.info.run_id,
    )

display(results_km.round(4))

,silhouette,davies_bouldin,calinski_harabasz,wcss
k,,,,
2,0.1394,2.3337,14396.2397,727966.6114
3,0.1563,1.7562,14977.2709,636116.1644
4,0.1951,1.6536,15828.6256,556932.5175
5,0.2060,1.5104,15841.9553,500490.8768
6,0.2151,1.4343,15895.9308,453831.5316
7,0.2145,1.3485,15623.1283,419240.4839
8,0.2247,1.2597,15607.8048,387132.0640
9,0.1792,1.3485,14947.6968,368354.3544
10,0.1873,1.3955,14490.2140,350523.2734


In [5]:
fig_elbow_km = plot_elbow(results_km, algo_name='KMeans', highlight_k=5)
plt.show()

fig_sil_km = plot_silhouette_comparison(results_km, algorithm_name='KMeans', highlight_k=5)
plt.show()

# ── Sélection k KMeans ────────────────────────────────────────────────────────
# Ajuster KMEANS_K après inspection visuelle de l'elbow et du pic silhouette
KMEANS_K = int(results_km['silhouette'].idxmax())
# Contraindre dans la plage 4–6 (contrainte projet)
if KMEANS_K < 4 or KMEANS_K > 10:
    KMEANS_K = results_km.loc[4:10, 'silhouette'].idxmax()
    print(f'k contraint à {KMEANS_K} (plage 4–10)')

print(f'k optimal KMeans : {KMEANS_K}')
print(f'  Silhouette     : {results_km.loc[KMEANS_K, "silhouette"]:.4f}')
print(f'  Davies-Bouldin : {results_km.loc[KMEANS_K, "davies_bouldin"]:.4f}')
print(f'  WCSS           : {results_km.loc[KMEANS_K, "wcss"]:,.0f}')

k optimal KMeans : 8
  Silhouette     : 0.2247
  Davies-Bouldin : 1.2597
  WCSS           : 387,132


## Section 2 — Bisecting K-Means : Variante Divisive

**Différence avec K-Means standard :** BisectingKMeans sélectionne récursivement le *plus grand cluster* et le divise en 2. Ce processus divisif produit naturellement des clusters **plus équilibrés en taille** — avantage important lorsqu'un segment (ex: churn) représente 70% des clients et risque d'absorber plusieurs centroïdes K-Means standard.

**Comparaison directe avec K-Means :** à même k, lequel a un meilleur silhouette/DB ? Si BisectingKMeans gagne, les données ont une structure naturellement hiérarchique.

In [6]:
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name='bisecting_kmeans_k_search') as parent_run:
    mlflow.set_tag('algorithm', 'bisecting_kmeans')
    results_bkm = run_bisecting_kmeans_search(
        X, k_range=range(2, 13), n_init=10,
        parent_run_id=parent_run.info.run_id,
    )

BKM_K = int(results_bkm['silhouette'].idxmax())
if BKM_K < 4 or BKM_K > 6:
    BKM_K = results_bkm.loc[4:6, 'silhouette'].idxmax()
print(f'k optimal BisectingKMeans : {BKM_K}')

display(results_bkm.round(4))

k optimal BisectingKMeans : 4


,silhouette,davies_bouldin,calinski_harabasz,wcss
k,,,,
2,0.1405,2.3300,14397.9819,727953.5340
3,0.1451,2.0690,13246.2908,654489.8100
4,0.1666,1.7026,14211.3140,576801.9200
5,0.1659,1.6613,13185.1228,536897.8573
6,0.1486,1.6588,12240.1403,507506.3324
7,0.1315,1.7559,11805.6717,477727.3620
8,0.1373,1.8508,11122.3799,458129.6330
9,0.1302,1.7991,10668.9013,438914.0185
10,0.1252,1.7337,10274.3159,422099.3935


In [ ]:
import gc
import numpy as np

# Libérer la mémoire avant le plot
plt.close('all')
gc.collect()

# Nettoyer les DataFrames — remplacer inf et NaN avant de plotter
def _clean_results(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace([np.inf, -np.inf], np.nan).dropna(how='all')

fig_centroid_compare = plot_metrics_comparison(
    {
        'KMeans':         _clean_results(results_km),
        'BisectingKMeans': _clean_results(results_bkm),
    },
    metrics=['silhouette', 'davies_bouldin', 'calinski_harabasz'],
)
plt.show()


## Section 3 — CAH (Clustering Hiérarchique Ascendant) — Ward Linkage

**Pourquoi Ward ?** Le critère Ward minimise la variance intra-cluster à chaque fusion — c'est l'analogue hiérarchique de la fonction objectif K-Means. Les résultats sont directement comparables.

**Lecture du dendrogramme :** Les **grandes barres verticales** indiquent des fusions entre groupes très différents — ce sont les frontières naturelles. La hauteur à laquelle on coupe le dendrogramme détermine k.

> ⚠️ **Note mémoire :** Le dendrogramme est calculé sur un échantillon de 15 000 clients (scipy linkage = O(n²) RAM sur la totalité). Les métriques silhouette/DB/CH sont calculées sur l'ensemble complet via sklearn AgglomerativeClustering.

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name='cah_k_search') as parent_run:
    mlflow.set_tag('algorithm', 'cah')
    mlflow.set_tag('linkage', 'ward')
    results_cah, Z_linkage = run_cah_search(
        X, k_range=range(2, 13), linkage_method='ward',
        dendrogram_sample_size=15_000,
        parent_run_id=parent_run.info.run_id,
    )

CAH_K = int(results_cah['silhouette'].idxmax())
if CAH_K < 4 or CAH_K > 6:
    CAH_K = results_cah.loc[4:6, 'silhouette'].idxmax()
print(f'k optimal CAH : {CAH_K}')

display(results_cah.round(4))

In [ ]:
fig_dendro = plot_dendrogram(Z_linkage, truncate_mode='lastp', p=30, highlight_k=CAH_K)
plt.show()

fig_hier_compare = plot_metrics_comparison(
    {'KMeans': results_km, 'CAH': results_cah},
    metrics=['silhouette', 'davies_bouldin', 'calinski_harabasz'],
)
plt.show()

## Section 4 — GMM (Gaussian Mixture Models) — BIC/AIC

**Pourquoi GMM ?** K-Means suppose des clusters **sphériques et de variance égale**. GMM (`covariance_type='full'`) modélise des clusters **ellipsoïdaux** avec des matrices de covariance distinctes — bien adapté si certains segments ont des formes allongées (ex: high-recency × high-monetary).

**BIC vs AIC :** Le BIC pénalise davantage la complexité — `k optimal = argmin(BIC)`. Si GMM choisit un k différent de K-Means, les clusters sont ellipsoïdaux (non sphériques).

**`max_soft_prob` :** Probabilité maximale d'appartenance par client, moyennée. Proche de 1.0 = clusters bien séparés. Proche de 1/k = chevauchement fort.

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name='gmm_k_search') as parent_run:
    mlflow.set_tag('algorithm', 'gmm')
    mlflow.set_tag('covariance_type', 'full')
    results_gmm = run_gmm_search(
        X, k_range=range(2, 13), covariance_type='full', n_init=5,
        parent_run_id=parent_run.info.run_id,
    )

GMM_K = int(results_gmm['bic'].idxmin())
if GMM_K < 4 or GMM_K > 6:
    GMM_K = results_gmm.loc[4:6, 'bic'].idxmin()
print(f'k optimal GMM (min BIC) : {GMM_K}')

display(results_gmm.round(4))

In [ ]:
fig_bic = plot_bic_aic(results_gmm, highlight_k=GMM_K)
plt.show()

fig_sil_gmm = plot_silhouette_comparison(results_gmm, algorithm_name='GMM', highlight_k=GMM_K)
plt.show()

# Comparaison silhouette KMeans vs GMM à même k
if KMEANS_K in results_gmm.index:
    delta = results_gmm.loc[KMEANS_K, 'silhouette'] - results_km.loc[KMEANS_K, 'silhouette']
    print(f'\nComparaison KMeans vs GMM à k={KMEANS_K} :')
    print(f'  KMeans silhouette : {results_km.loc[KMEANS_K, "silhouette"]:.4f}')
    print(f'  GMM    silhouette : {results_gmm.loc[KMEANS_K, "silhouette"]:.4f}')
    print(f'  Δ = {delta:+.4f}  {"→ Clusters ellipsoïdaux" if delta > 0.01 else "→ Clusters sphériques (KMeans suffisant)"}')

## Section 5 — DBSCAN : Estimation eps + Grid Search

**Principe :** DBSCAN classe comme *core points* les points ayant au moins `min_samples` voisins dans un rayon `eps`. Les points non-atteignables = **bruit (label -1)**.

**Estimation eps :** Le k-distance graph trie les distances au k-ème voisin par ordre croissant. L'inflexion indique l'eps naturel. On utilise le **p95** pour une estimation robuste sans sélection manuelle.

**Résultat attendu sur Olist :** Les données RFM continues tendent à former peu de clusters très denses + un large nuage de bruit. DBSCAN capture ce profil si eps est bien calibré.

In [ ]:
MIN_SAMPLES_DBSCAN = 10
recommended_eps, fig_kdist = estimate_dbscan_eps(X, min_samples=MIN_SAMPLES_DBSCAN, percentile=95.0)
plt.show()
print(f'eps recommandé (p95) : {recommended_eps:.4f}')

eps_grid = [round(recommended_eps * f, 4) for f in [0.5, 0.7, 1.0, 1.3, 1.7]]
min_samples_grid = [5, 10, 18]
print(f'eps_grid    : {eps_grid}')
print(f'min_samples : {min_samples_grid}')

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name='dbscan_grid_search') as parent_run:
    mlflow.set_tag('algorithm', 'dbscan')
    results_db = run_dbscan_search(
        X, eps_values=eps_grid, min_samples_values=min_samples_grid,
        parent_run_id=parent_run.info.run_id,
    )

valid_db = results_db[
    results_db['n_clusters'].between(2, 10) &
    (results_db['noise_ratio'] < 0.30)
].sort_values('silhouette', ascending=False)

DBSCAN_VALID = len(valid_db) > 0
print(f'Configurations DBSCAN valides (k∈[2,10], noise<30%) : {len(valid_db)}')

if DBSCAN_VALID:
    best_db_row = valid_db.iloc[0]
    EPS_BEST = float(best_db_row['eps'])
    MS_BEST  = int(best_db_row['min_samples'])
    print(f'Meilleure config : eps={EPS_BEST}, min_samples={MS_BEST}')
    print(f'  n_clusters={int(best_db_row["n_clusters"])}, noise={best_db_row["noise_ratio"]:.1%}')
else:
    EPS_BEST, MS_BEST = recommended_eps, MIN_SAMPLES_DBSCAN
    print('⚠️  Aucune config valide — typique sur données RFM continues.')
    print('   DBSCAN inclus dans la comparaison, non éligible pour la sélection finale.')

display(results_db.head(10).round(4))

## Section 6 — HDBSCAN : Robustesse aux Densités Variables

**HDBSCAN vs DBSCAN :** HDBSCAN (Hierarchical DBSCAN) construit une hiérarchie de densité et sélectionne les clusters les plus stables — sans eps fixe. Seul `min_cluster_size` est critique :
- Trop petit → clusters bruités, nombreux et instables
- Trop grand → sur-fusion, perd des segments réels

**Avantage sur Olist :** Les données géographiques créent des densités variables — certains états ont beaucoup de clients (SP), d'autres très peu (AM). HDBSCAN gère naturellement cette hétérogénéité là où DBSCAN échoue avec un eps fixe.

**`mean_membership_prob` :** Probabilité d'appartenance moyenne (soft memberships). Proche de 1.0 = assignations confiantes.

In [ ]:
try:
    from sklearn.cluster import HDBSCAN
    HDBSCAN_AVAILABLE = True
    print(f'✅ HDBSCAN disponible (sklearn {sklearn.__version__})')
except ImportError:
    HDBSCAN_AVAILABLE = False
    print(f'⚠️  HDBSCAN non disponible (sklearn {sklearn.__version__} < 1.3) — section ignorée')

HDBSCAN_VALID = False
results_hdb   = pd.DataFrame()
MCS_BEST, MS_HDB_INT = 200, None

In [ ]:
if HDBSCAN_AVAILABLE:
    if mlflow.active_run():
        mlflow.end_run()

    with mlflow.start_run(run_name='hdbscan_search') as parent_run:
        mlflow.set_tag('algorithm', 'hdbscan')
        results_hdb = run_hdbscan_search(
            X,
            min_cluster_sizes=[50, 100, 200, 500, 1000],
            min_samples_list=[5, 10, None],
            parent_run_id=parent_run.info.run_id,
        )

    valid_hdb = results_hdb[
        results_hdb['n_clusters'].between(2, 10) &
        (results_hdb['noise_ratio'] < 0.30)
    ].sort_values('silhouette', ascending=False)

    HDBSCAN_VALID = len(valid_hdb) > 0
    print(f'Configurations HDBSCAN valides : {len(valid_hdb)}')

    if HDBSCAN_VALID:
        best_hdb_row = valid_hdb.iloc[0]
        MCS_BEST    = int(best_hdb_row['min_cluster_size'])
        ms_val      = best_hdb_row['min_samples']
        MS_HDB_INT  = None if str(ms_val) == 'auto' else int(ms_val)
        print(f'Meilleure config : min_cluster_size={MCS_BEST}, min_samples={ms_val}')
        print(f'  n_clusters={int(best_hdb_row["n_clusters"])}, noise={best_hdb_row["noise_ratio"]:.1%}')
    else:
        print('⚠️  Aucune config valide — HDBSCAN inclus dans comparaison, non éligible.')

    display(results_hdb.head(10).round(4))

    # Comparaison noise_ratio DBSCAN vs HDBSCAN
    if not results_db.empty:
        print(f'\n── Noise ratio : DBSCAN min={results_db["noise_ratio"].min():.1%}'
              f' vs HDBSCAN min={results_hdb["noise_ratio"].min():.1%}')

## Section 7 — Fits Finaux avec MLFlow (6 Algorithmes)

Chaque algorithme reçoit son run MLFlow top-level avec le k sélectionné. Les modèles avec `predict()` sont loggés comme artefacts MLFlow. DBSCAN et HDBSCAN ne sont fitté que si leur configuration est valide.

In [ ]:
all_labels  = {}  # algo_name → labels array
all_models  = {}  # algo_name → fitted model
all_metrics = {}  # algo_name → metrics dict

if mlflow.active_run():
    mlflow.end_run()

# ── K-Means final ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name=f'kmeans_final_k{KMEANS_K}'):
    mlflow.set_tag('run_type', 'final')
    model, labels, metrics = fit_kmeans(X, k=KMEANS_K, n_init=20)
    mlflow.log_figure(fig_elbow_km, 'elbow_kmeans.png')
    fig_pca_km = plot_pca_clusters(X, labels, f'KMeans k={KMEANS_K} — PCA 2D')
    mlflow.log_figure(fig_pca_km, 'pca_kmeans.png')
    plt.show()
all_models['KMeans']  = model
all_labels['KMeans']  = labels
all_metrics['KMeans'] = metrics
print(f'✅ KMeans k={KMEANS_K} : silhouette={metrics["silhouette"]:.4f}')

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

# ── Bisecting K-Means final ───────────────────────────────────────────────────
with mlflow.start_run(run_name=f'bisecting_kmeans_final_k{BKM_K}'):
    mlflow.set_tag('run_type', 'final')
    model, labels, metrics = fit_bisecting_kmeans(X, k=BKM_K)
    fig_pca_bkm = plot_pca_clusters(X, labels, f'BisectingKMeans k={BKM_K} — PCA 2D')
    mlflow.log_figure(fig_pca_bkm, 'pca_bisecting_kmeans.png')
    plt.show()
all_models['BisectingKMeans']  = model
all_labels['BisectingKMeans']  = labels
all_metrics['BisectingKMeans'] = metrics
print(f'✅ BisectingKMeans k={BKM_K} : silhouette={metrics["silhouette"]:.4f}')

if mlflow.active_run():
    mlflow.end_run()

# ── CAH final ─────────────────────────────────────────────────────────────────
with mlflow.start_run(run_name=f'cah_final_k{CAH_K}'):
    mlflow.set_tag('run_type', 'final')
    model, labels, metrics = fit_cah(X, k=CAH_K)
    mlflow.log_figure(fig_dendro, 'dendrogram_cah.png')
    fig_pca_cah = plot_pca_clusters(X, labels, f'CAH Ward k={CAH_K} — PCA 2D')
    mlflow.log_figure(fig_pca_cah, 'pca_cah.png')
    plt.show()
all_models['CAH']  = model
all_labels['CAH']  = labels
all_metrics['CAH'] = metrics
print(f'✅ CAH k={CAH_K} : silhouette={metrics["silhouette"]:.4f}')

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

# ── GMM final ─────────────────────────────────────────────────────────────────
with mlflow.start_run(run_name=f'gmm_final_k{GMM_K}'):
    mlflow.set_tag('run_type', 'final')
    model, labels, metrics = fit_gmm(X, k=GMM_K, covariance_type='full')
    mlflow.log_figure(fig_bic, 'bic_aic_gmm.png')
    fig_pca_gmm = plot_pca_clusters(X, labels, f'GMM k={GMM_K} — PCA 2D')
    mlflow.log_figure(fig_pca_gmm, 'pca_gmm.png')
    plt.show()
all_models['GMM']  = model
all_labels['GMM']  = labels
all_metrics['GMM'] = metrics
print(f'✅ GMM k={GMM_K} : silhouette={metrics["silhouette"]:.4f} | BIC={metrics["bic"]:,.0f} | soft_prob={metrics["max_soft_prob"]:.3f}')

# ── DBSCAN final (conditionnel) ───────────────────────────────────────────────
if DBSCAN_VALID:
    if mlflow.active_run(): mlflow.end_run()
    with mlflow.start_run(run_name=f'dbscan_final_eps{EPS_BEST:.3f}'):
        mlflow.set_tag('run_type', 'final')
        model, labels, metrics = fit_dbscan(X, eps=EPS_BEST, min_samples=MS_BEST)
        fig_pca_db = plot_pca_clusters(X, labels, f'DBSCAN eps={EPS_BEST:.3f} — PCA 2D')
        mlflow.log_figure(fig_pca_db, 'pca_dbscan.png')
        plt.show()
    all_labels['DBSCAN']  = labels
    all_metrics['DBSCAN'] = metrics
    print(f'✅ DBSCAN : silhouette={metrics["silhouette"]:.4f} | k={metrics["n_clusters"]} | noise={metrics["noise_ratio"]:.1%}')

# ── HDBSCAN final (conditionnel) ──────────────────────────────────────────────
if HDBSCAN_VALID:
    if mlflow.active_run(): mlflow.end_run()
    with mlflow.start_run(run_name=f'hdbscan_final_mcs{MCS_BEST}'):
        mlflow.set_tag('run_type', 'final')
        model, labels, metrics = fit_hdbscan(X, min_cluster_size=MCS_BEST, min_samples=MS_HDB_INT)
        fig_pca_hdb = plot_pca_clusters(X, labels, f'HDBSCAN mcs={MCS_BEST} — PCA 2D')
        mlflow.log_figure(fig_pca_hdb, 'pca_hdbscan.png')
        plt.show()
    all_labels['HDBSCAN']  = labels
    all_metrics['HDBSCAN'] = metrics
    print(f'✅ HDBSCAN : silhouette={metrics["silhouette"]:.4f} | k={metrics["n_clusters"]} | noise={metrics["noise_ratio"]:.1%}')

if mlflow.active_run():
    mlflow.end_run()

## Section 8 — Comparaison des Algorithmes & Sélection du Meilleur Modèle

**Score composite :** Normalisation [0,1] de chaque métrique × direction (silhouette ↑, Davies-Bouldin ↓, Calinski-Harabasz ↑), puis moyenne. Ce score résume la qualité globale sans biais d'échelle.

**Règle de sélection :** Seuls les algorithmes **éligibles pour la production** (avec `predict()`) participent : KMeans, BisectingKMeans, CAH, GMM. DBSCAN et HDBSCAN sont comparés mais non éligibles (transductifs).

In [ ]:
# ── Construction du tableau comparatif ────────────────────────────────────────
comparison_rows = []
for algo, metrics in all_metrics.items():
    row = {'algorithm': algo}
    row.update({k: v for k, v in metrics.items() if not isinstance(v, dict)})
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index('algorithm')

# Score composite sur métriques partagées
shared = ['silhouette', 'davies_bouldin', 'calinski_harabasz']
comp   = comparison_df[shared].copy().dropna()

sil_norm = (comp['silhouette'] - comp['silhouette'].min()) / (comp['silhouette'].max() - comp['silhouette'].min() + 1e-9)
db_norm  = (comp['davies_bouldin'] - comp['davies_bouldin'].min()) / (comp['davies_bouldin'].max() - comp['davies_bouldin'].min() + 1e-9)
ch_norm  = (comp['calinski_harabasz'] - comp['calinski_harabasz'].min()) / (comp['calinski_harabasz'].max() - comp['calinski_harabasz'].min() + 1e-9)
comparison_df.loc[comp.index, 'composite_score'] = (sil_norm - db_norm + ch_norm) / 3

display_cols = [c for c in ['silhouette', 'davies_bouldin', 'calinski_harabasz',
                             'wcss', 'bic', 'n_clusters', 'noise_ratio', 'composite_score']
                if c in comparison_df.columns]

print('\n══════════════════════════════════════════════════')
print('  COMPARAISON GLOBALE — 6 ALGORITHMES')
print('══════════════════════════════════════════════════')
display(
    comparison_df[display_cols].round(4).style
    .highlight_max(subset=['silhouette', 'calinski_harabasz', 'composite_score'], color='#d4edda')
    .highlight_min(subset=['davies_bouldin'], color='#d4edda')
)

In [ ]:
# ── Sélection du meilleur modèle (parmi les éligibles) ────────────────────────
ELIGIBLE_ALGOS = ['KMeans', 'BisectingKMeans', 'CAH', 'GMM']
eligible_in_df = [
    a for a in ELIGIBLE_ALGOS
    if a in comparison_df.index
    and 'composite_score' in comparison_df.columns
    and not pd.isna(comparison_df.loc[a, 'composite_score'])
]

BEST_ALGORITHM = comparison_df.loc[eligible_in_df, 'composite_score'].idxmax()
BEST_LABELS    = all_labels[BEST_ALGORITHM]
BEST_MODEL     = all_models[BEST_ALGORITHM]
BEST_K         = {'KMeans': KMEANS_K, 'BisectingKMeans': BKM_K,
                  'CAH': CAH_K, 'GMM': GMM_K}.get(BEST_ALGORITHM, KMEANS_K)
BEST_SCORE     = float(comparison_df.loc[BEST_ALGORITHM, 'composite_score'])

print(f'\n🏆 Meilleur algorithme : {BEST_ALGORITHM}')
print(f'   k = {BEST_K}')
print(f'   Score composite    = {BEST_SCORE:.4f}')
print(f'   Silhouette         = {comparison_df.loc[BEST_ALGORITHM, "silhouette"]:.4f}')
print(f'   Davies-Bouldin     = {comparison_df.loc[BEST_ALGORITHM, "davies_bouldin"]:.4f}')
print(f'\n   DBSCAN / HDBSCAN : comparés, non éligibles (pas de predict() pour nouveaux clients).')

## Section 9 — Profiling des Clusters (Meilleur Modèle)

**Interprétation en valeurs brutes :** Le profiling utilise les médianes des features **non normalisées** pour que les chiffres soient interprétables en unités métier : Recency en jours, Monetary en BRL.

**Nommage marketing :** Chaque cluster est nommé selon son profil dominant. Exemples types :
- Haute Recency + faible Monetary → **"Dormants"**
- Faible Recency + haute Monetary + F≥2 → **"Champions"**
- Faible Recency + faible Monetary + F=1 → **"Acheteurs Budget"**

In [ ]:
# ── Attacher les labels au dataframe raw ─────────────────────────────────────
df_labeled = df_raw.copy()
df_labeled['cluster'] = pd.Series(BEST_LABELS, index=X_scaled_df.index).values
assert df_labeled['cluster'].isna().sum() == 0, 'Labels manquants — vérifier alignement'

print(f'Dataset labellisé : {df_labeled.shape}')
size_dist = df_labeled['cluster'].value_counts().sort_index()
for c, n in size_dist.items():
    pct = n / len(df_labeled) * 100
    bar = '█' * int(pct / 2)
    print(f'  Cluster {c} : {n:>7,} clients ({pct:>5.1f}%)  {bar}')

In [ ]:
# ── Profil des clusters (médianes brutes) ────────────────────────────────────
RAW_PROFILE_FEATURES = [c for c in [
    'Recency', 'Monetary', 'Frequency',
    'avg_freight_ratio', 'avg_delivery_delay', 'avg_review_score',
    'payment_type_cc_flag', 'avg_installments', 'region_freight_score',
] if c in df_labeled.columns]

profile_df = build_cluster_profile(
    df_labeled, BEST_LABELS, feature_cols=RAW_PROFILE_FEATURES,
)

fmt = {c: '{:.2f}' for c in RAW_PROFILE_FEATURES}
fmt.update({'CLV_proxy': '{:.1f}', 'Recency': '{:.0f}', 'Monetary': '{:.1f}',
            'pct_customers': '{:.1f}%', 'n_customers': '{:,.0f}'})

display(
    profile_df.style
    .format({k: v for k, v in fmt.items() if k in profile_df.columns})
    .background_gradient(cmap='YlOrRd', subset=['CLV_proxy'] if 'CLV_proxy' in profile_df.columns else [])
)

In [ ]:
# ── Heatmap des centroids scaled ─────────────────────────────────────────────
scaled_centroids = (
    pd.DataFrame(X, columns=FINAL_FEATURES, index=X_scaled_df.index)
    .assign(cluster=BEST_LABELS)
    .groupby('cluster')[FINAL_FEATURES]
    .mean()
)
fig_heatmap = plot_cluster_heatmap(scaled_centroids)
plt.show()

# ── Radar chart (médianes brutes normalisées [0,1]) ──────────────────────────
RADAR_FEATURES = [c for c in ['Recency', 'Monetary', 'avg_freight_ratio',
                               'avg_delivery_delay', 'avg_review_score', 'avg_installments']
                  if c in profile_df.columns]
fig_radar = plot_radar_chart(profile_df, feature_cols=RADAR_FEATURES)
plt.show()

# ── PCA 2D final ─────────────────────────────────────────────────────────────
fig_pca_final = plot_pca_clusters(
    X, BEST_LABELS,
    title=f'{BEST_ALGORITHM} k={BEST_K} — Projection PCA 2D (vue finale)',
)
plt.show()

# ── Distribution en barplot ───────────────────────────────────────────────────
fig_bar, ax = plt.subplots(figsize=(8, 4))
pcts = df_labeled['cluster'].value_counts(normalize=True).sort_index() * 100
palette = sns.color_palette('tab10', len(pcts))
ax.bar([str(c) for c in pcts.index], pcts.values, color=palette)
for i, (c, v) in enumerate(pcts.items()):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)
ax.set_title(f'{BEST_ALGORITHM} k={BEST_K} — Distribution des clusters', fontsize=12)
ax.set_xlabel('Cluster')
ax.set_ylabel('% clients')
plt.tight_layout()
plt.show()

## Section 10 — Validation des Hypothèses H1–H6

Les 6 hypothèses formulées en EDA sont maintenant testables quantitativement contre les profils de clusters.

| ID | Hypothèse | Condition de validation |
|----|-----------|------------------------|
| H1 | High-spenders = acheteurs peu fréquents | Cluster max-CLV avec Frequency ≤ 1.5 |
| H2 | Recency > 300j = churn effectif | ≥ 1 cluster avec Recency médiane > 300j |
| H3 | Géographie prédit le freight burden | Spearman ρ(region_score, freight_ratio) > 0 |
| H4 | Boleto = proxy faible revenu → low-value | Cluster min_CC_flag < Monetary médiane |
| H5 | Délai livraison varie entre segments | Range délai inter-clusters > 5 jours |
| H6 | Clients fidèles (F≥2) forment un cluster distinct | ≥ 1 cluster avec F ≥ 2 ET Monetary > médiane |

In [ ]:
validation_df = validate_hypotheses(df_labeled, profile_df)

display(
    validation_df.set_index('hypothesis_id').style
    .applymap(
        lambda v: 'background-color: #d4edda' if v == 'VALIDATED'
        else ('background-color: #fff3cd' if v == 'PARTIAL'
              else 'background-color: #f8d7da'),
        subset=['status']
    )
    .set_properties(**{'text-align': 'left', 'font-size': '11px'})
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
)

n_validated = (validation_df['status'] == 'VALIDATED').sum()
n_partial   = (validation_df['status'] == 'PARTIAL').sum()
print(f'\nRésultat : {n_validated}/6 VALIDÉES, {n_partial}/6 PARTIELLES')

In [ ]:
# ── Détail des segments clés ──────────────────────────────────────────────────
print('═══ H2 — Segment Dormants (Recency > 300j) ═══')
if 'Recency' in profile_df.columns:
    churn_seg = profile_df[profile_df['Recency'] > 300]
    if len(churn_seg) > 0:
        print(churn_seg[['cluster', 'n_customers', 'pct_customers', 'Recency', 'Monetary']].to_string(index=False))
    else:
        print(f'  Recency max parmi les clusters : {profile_df["Recency"].max():.0f} jours')

print('\n═══ H6 — Segment Champions (F≥2 + Monetary élevé) ═══')
if 'Frequency' in profile_df.columns and 'Monetary' in profile_df.columns:
    global_med = df_labeled['Monetary'].median() if 'Monetary' in df_labeled.columns else profile_df['Monetary'].median()
    champions  = profile_df[(profile_df['Frequency'] >= 2) & (profile_df['Monetary'] > global_med)]
    if len(champions) > 0:
        print(champions[['cluster', 'n_customers', 'pct_customers', 'Frequency', 'Monetary', 'CLV_proxy']].to_string(index=False))
    else:
        print(f'  Frequency max : {profile_df["Frequency"].max():.2f} | Monetary max : {profile_df["Monetary"].max():.1f} BRL')

print('\n═══ H3 — Géographie × Freight ═══')
if 'region_freight_score' in profile_df.columns and 'avg_freight_ratio' in profile_df.columns:
    print(profile_df[['cluster', 'region_freight_score', 'avg_freight_ratio']].sort_values('region_freight_score').to_string(index=False))

## Section 11 — Export & Handoff Production

Trois artefacts sont exportés :
1. **`models/best_clustering_*.pkl`** + sidecar JSON — modèle + métadonnées pour l'inférence dashboard
2. **`data/processed/customer_features_labeled.parquet`** — tous les clients avec leur `cluster` label
3. **`data/processed/cluster_profile.parquet`** — profil médian par cluster pour les visualisations dashboard

Un run MLFlow `best_model_export` centralise tous les artefacts finaux.

In [ ]:
# ── Sauvegarde du modèle ──────────────────────────────────────────────────────
MODEL_FILENAME = f'best_clustering_{BEST_ALGORITHM.lower()}_k{BEST_K}.pkl'
MODEL_PATH     = MODELS_DIR / MODEL_FILENAME

save_model(
    BEST_MODEL, MODEL_PATH,
    metadata={
        'algorithm':          BEST_ALGORITHM,
        'k':                  BEST_K,
        'features':           FINAL_FEATURES,
        'silhouette':         float(comparison_df.loc[BEST_ALGORITHM, 'silhouette']),
        'davies_bouldin':     float(comparison_df.loc[BEST_ALGORITHM, 'davies_bouldin']),
        'calinski_harabasz':  float(comparison_df.loc[BEST_ALGORITHM, 'calinski_harabasz']),
        'composite_score':    BEST_SCORE,
    }
)

# ── Export parquets ───────────────────────────────────────────────────────────
LABELED_PATH = PROCESSED_DIR / 'customer_features_labeled.parquet'
PROFILE_PATH = PROCESSED_DIR / 'cluster_profile.parquet'

df_labeled.to_parquet(LABELED_PATH, index=False)
profile_df.to_parquet(PROFILE_PATH, index=False)

print(f'✅ Exportés :')
print(f'   {MODEL_PATH}')
print(f'   {LABELED_PATH}')
print(f'   {PROFILE_PATH}')

In [ ]:
# ── Log MLFlow final (tous les artefacts) ─────────────────────────────────────
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name=f'best_model_export_{BEST_ALGORITHM.lower()}') as final_run:
    mlflow.set_tag('run_type', 'export')
    mlflow.set_tag('algorithm', BEST_ALGORITHM)
    mlflow.log_param('k', BEST_K)
    mlflow.log_param('features', str(FINAL_FEATURES))
    mlflow.log_metrics({
        'silhouette':         float(comparison_df.loc[BEST_ALGORITHM, 'silhouette']),
        'davies_bouldin':     float(comparison_df.loc[BEST_ALGORITHM, 'davies_bouldin']),
        'calinski_harabasz':  float(comparison_df.loc[BEST_ALGORITHM, 'calinski_harabasz']),
        'composite_score':    BEST_SCORE,
    })
    mlflow.log_artifact(str(MODEL_PATH))
    mlflow.log_artifact(str(LABELED_PATH))
    mlflow.log_artifact(str(PROFILE_PATH))
    mlflow.log_figure(fig_heatmap,   'final_cluster_heatmap.png')
    mlflow.log_figure(fig_radar,     'final_radar_chart.png')
    mlflow.log_figure(fig_pca_final, 'final_pca_clusters.png')

mlflow.end_run()
print(f'✅ Artefacts loggés dans MLFlow run : best_model_export_{BEST_ALGORITHM.lower()}')

In [ ]:
# ── Test de rechargement (sanity check production) ────────────────────────────
reloaded_model = load_model(MODEL_PATH)
test_labels    = reloaded_model.predict(X[:10])
print(f'✅ Sanity check : model.predict(X[:10]) = {test_labels}')

# ── Récapitulatif final ────────────────────────────────────────────────────────
print(f"""
══════════════════════════════════════════════════════════════
  NOTEBOOK 03 — RÉCAPITULATIF FINAL
══════════════════════════════════════════════════════════════
  Algorithmes comparés  : KMeans, BisectingKMeans, CAH, GMM
                          + DBSCAN ({'valide' if DBSCAN_VALID else 'non valide'})
                          + HDBSCAN ({'valide' if HDBSCAN_VALID else 'non valide'})
  Meilleur algorithme   : {BEST_ALGORITHM} (k={BEST_K})
  Score composite       : {BEST_SCORE:.4f}
  Silhouette            : {comparison_df.loc[BEST_ALGORITHM, 'silhouette']:.4f}
  Clients labelisés     : {len(df_labeled):,}
  Hypothèses validées   : {n_validated}/6 VALIDÉES, {n_partial}/6 PARTIELLES
══════════════════════════════════════════════════════════════
  Prochain notebook : 04_simulation.ipynb
""")